# 🔐 UEBA Lab — Notebook 01: Data Ingestion & Feature Engineering

**Dataset:** CMU CERT Insider Threat Dataset r4.2 (`r4.2.tar.bz2`)
**Goal:** Load all event logs, explore behavioral patterns, and build a behavioral feature matrix (users × days × features).

## 📋 Notebook Sections
| Section | Topic |
|---------|-------|
| 1.0 | Setup & Configuration |
| 1.1 | Extract & Load CERT r4.2 CSVs |
| 1.2 | LDAP Enrichment |
| 1.3 | Exploratory Data Analysis |
| 1.4 | Feature Engineering |
| 1.5 | Save Feature Matrix |

> **⚠️ Before running:** Place `r4.2.tar.bz2` in the same directory as this notebook.

---
## 1.0 Setup & Configuration

In [ ]:
import os
import tarfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Configuration ────────────────────────────────────────────
ARCHIVE_NAME        = 'r4.2.tar.bz2'
EXTRACT_DIR         = './cert_r42'
BASELINE_WINDOW_DAYS = 30

SENSITIVE_KEYWORDS = [
    'model', 'proprietary', 'confidential', 'secret',
    'patent', 'source', 'algorithm', 'key', 'credential',
    'private', 'internal', 'classified', 'restricted'
]

JOB_SITE_DOMAINS = [
    'linkedin.com', 'indeed.com', 'glassdoor.com',
    'monster.com', 'careerbuilder.com', 'ziprecruiter.com'
]

CLOUD_STORAGE_DOMAINS = [
    'dropbox.com', 'drive.google.com', 'onedrive.live.com',
    'box.com', 'mega.nz', 'wetransfer.com'
]

WEBMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com',
    'outlook.com', 'protonmail.com'
]

print('✅ Configuration loaded.')
print(f'   Archive          : {ARCHIVE_NAME}')
print(f'   Extract target   : {EXTRACT_DIR}')
print(f'   Baseline window  : {BASELINE_WINDOW_DAYS} days')

---
## 1.1 Extract & Load CERT r4.2 CSVs

We extract `r4.2.tar.bz2` in-place, then load all five event log files.
Each file shares a common `user` column that acts as the join key across sources.

In [ ]:
# ── Extract archive ───────────────────────────────────────────
if not os.path.exists(EXTRACT_DIR):
    print(f'📦 Extracting {ARCHIVE_NAME} → {EXTRACT_DIR} ...')
    with tarfile.open(ARCHIVE_NAME, 'r:bz2') as tar:
        tar.extractall(path=EXTRACT_DIR)
    print('✅ Extraction complete.')
else:
    print(f'✅ Archive already extracted at {EXTRACT_DIR}')

# ── Discover extracted layout ─────────────────────────────────
print('\n📂 Extracted contents:')
for root, dirs, fnames in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = '   ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in fnames:
        print(f'{indent}   {f}')

In [ ]:
# ── Locate the data root (handles nested subdirectory) ────────
def find_cert_root(base_dir: str) -> str:
    """
    Walk extracted directory to find the folder that contains
    logon.csv — handles any nesting depth.
    """
    for root, dirs, files in os.walk(base_dir):
        if 'logon.csv' in files:
            return root
    raise FileNotFoundError(
        f'logon.csv not found under {base_dir}. '
        f'Check that {ARCHIVE_NAME} is the correct r4.2 archive.'
    )

DATA_ROOT = find_cert_root(EXTRACT_DIR)
print(f'✅ CERT data root located: {DATA_ROOT}')

In [ ]:
# ── CSV loader helper ─────────────────────────────────────────
def load_cert_csv(filename: str, date_cols: list) -> pd.DataFrame:
    """
    Load a CERT CSV, parse date columns, and standardise
    the user column to 'user_id'.
    """
    filepath = os.path.join(DATA_ROOT, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'❌ Not found: {filepath}')
    df = pd.read_csv(filepath, parse_dates=date_cols, low_memory=False)
    if 'user' in df.columns:
        df.rename(columns={'user': 'user_id'}, inplace=True)
    print(f'   ✅ {filename:<20} {len(df):>10,} rows  '
          f'cols: {list(df.columns)}')
    return df

print('📂 Loading CERT r4.2 event logs...\n')
logon  = load_cert_csv('logon.csv',  date_cols=['date'])
device = load_cert_csv('device.csv', date_cols=['date'])
files  = load_cert_csv('file.csv',   date_cols=['date'])
email  = load_cert_csv('email.csv',  date_cols=['date'])
http   = load_cert_csv('http.csv',   date_cols=['date'])

total_events = len(logon) + len(device) + len(files) + len(email) + len(http)
print(f'\n📊 Total events loaded: {total_events:,}')

---
## 1.2 LDAP Enrichment

CERT r4.2 ships with monthly LDAP snapshots under `LDAP/`.
We load all snapshots, deduplicate to one record per user
(keeping the most recent), and join `role` + `department`
onto every event log.

In [ ]:
# ── Load LDAP snapshots ───────────────────────────────────────
ldap_dir = os.path.join(DATA_ROOT, 'LDAP')
ldap_files = sorted([f for f in os.listdir(ldap_dir) if f.endswith('.csv')])
print(f'📂 Found {len(ldap_files)} LDAP snapshot files: {ldap_files}\n')

ldap_frames = []
for lf in ldap_files:
    tmp = pd.read_csv(os.path.join(ldap_dir, lf))
    tmp.columns = tmp.columns.str.lower().str.strip().str.replace(' ', '_')
    ldap_frames.append(tmp)

ldap_raw = pd.concat(ldap_frames, ignore_index=True)

# Standardise user column
for candidate in ['user_id', 'user', 'employee_id', 'name']:
    if candidate in ldap_raw.columns:
        ldap_raw.rename(columns={candidate: 'user_id'}, inplace=True)
        break

# Keep latest record per user
keep_cols = ['user_id']
for col in ['role', 'department', 'team', 'supervisor', 'o', 'title']:
    if col in ldap_raw.columns:
        keep_cols.append(col)

ldap = (
    ldap_raw[keep_cols]
    .drop_duplicates(subset=['user_id'], keep='last')
    .reset_index(drop=True)
)

# Normalise role/department column names for r4.2
if 'o' in ldap.columns and 'department' not in ldap.columns:
    ldap.rename(columns={'o': 'department'}, inplace=True)
if 'title' in ldap.columns and 'role' not in ldap.columns:
    ldap.rename(columns={'title': 'role'}, inplace=True)

print(f'✅ LDAP enrichment table: {len(ldap):,} unique users')
print(f'   Columns available: {list(ldap.columns)}')
if 'role' in ldap.columns:
    print(f'   Unique roles      : {ldap["role"].nunique()}')
if 'department' in ldap.columns:
    print(f'   Unique departments: {ldap["department"].nunique()}')
ldap.head(3)

In [ ]:
# ── Enrich all event logs with LDAP + derived time columns ────
def enrich(df: pd.DataFrame, ldap: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(ldap, on='user_id', how='left')
    df['date_only']      = df['date'].dt.normalize()
    df['hour']           = df['date'].dt.hour
    df['weekday']        = df['date'].dt.dayofweek
    df['is_weekend']     = df['weekday'] >= 5
    df['is_after_hours'] = ~df['hour'].between(8, 18)
    return df

logon  = enrich(logon,  ldap)
device = enrich(device, ldap)
files  = enrich(files,  ldap)
email  = enrich(email,  ldap)
http   = enrich(http,   ldap)

print('✅ LDAP enrichment + time columns applied to all event logs.')
print('\nSample enriched logon record:')
logon[['user_id', 'date', 'role', 'department',
       'hour', 'is_after_hours', 'is_weekend']].head(3)

---
## 1.3 Exploratory Data Analysis

Before building features we explore the behavioral landscape:
- Event volume over time per source
- Hourly activity heatmap
- Top users by file access volume
- After-hours login rate by role
- USB event distribution

In [ ]:
# ── Dataset summary ───────────────────────────────────────────
all_users = (
    set(logon['user_id']) | set(files['user_id']) |
    set(device['user_id']) | set(email['user_id']) | set(http['user_id'])
)
date_min = min(df['date'].min() for df in [logon, files, device, email, http])
date_max = max(df['date'].max() for df in [logon, files, device, email, http])

summary = pd.DataFrame({
    'Source'      : ['logon', 'device', 'files', 'email', 'http'],
    'Rows'        : [len(logon), len(device), len(files), len(email), len(http)],
    'Unique Users': [
        logon['user_id'].nunique(),  device['user_id'].nunique(),
        files['user_id'].nunique(),  email['user_id'].nunique(),
        http['user_id'].nunique()
    ],
    'Date Min': [df['date'].min().date()
                 for df in [logon, device, files, email, http]],
    'Date Max': [df['date'].max().date()
                 for df in [logon, device, files, email, http]],
})

print('=' * 60)
print(f'  Total unique users : {len(all_users):,}')
print(f'  Dataset date range : {date_min.date()} → {date_max.date()}')
print(f'  Total span         : {(date_max - date_min).days} days')
print(f'  Total events       : {summary["Rows"].sum():,}')
print('=' * 60)
summary

In [ ]:
# ── EDA Plot 1: Daily event volume by source ──────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Daily Event Volume by Source — CERT r4.2',
             fontsize=14, fontweight='bold', y=1.01)

sources = [
    (logon,  'Logon Events',  '#1E90FF'),
    (device, 'Device Events', '#FF6B35'),
    (files,  'File Events',   '#00C853'),
    (email,  'Email Events',  '#FFD700'),
    (http,   'HTTP Events',   '#DA70D6'),
]

for ax, (df, label, color) in zip(axes, sources):
    daily = df.groupby('date_only').size().reset_index(name='count')
    ax.fill_between(daily['date_only'], daily['count'],
                    alpha=0.4, color=color)
    ax.plot(daily['date_only'], daily['count'],
            color=color, linewidth=1.2)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.savefig('eda_event_volume.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Event volume timeline saved → eda_event_volume.png')

In [ ]:
# ── EDA Plot 2: Hourly activity heatmap ───────────────────────
hourly_data = pd.concat([
    logon[['hour', 'weekday']].assign(source='logon'),
    files[['hour', 'weekday']].assign(source='files'),
    http[['hour', 'weekday']].assign(source='http'),
])

pivot = (
    hourly_data
    .groupby(['weekday', 'hour'])
    .size()
    .reset_index(name='count')
    .pivot(index='weekday', columns='hour', values='count')
    .fillna(0)
)

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

plt.figure(figsize=(16, 4))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    yticklabels=day_labels,
    cbar_kws={'label': 'Event Count'},
)
plt.title('Activity Heatmap — Hour of Day × Day of Week (All Users)',
          fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.savefig('eda_hourly_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Hourly heatmap saved → eda_hourly_heatmap.png')
print('\n💡 Peak activity: 09:00–17:00 Mon–Fri.')
print('   After-hours events are rare → strong anomaly signal.')

In [ ]:
# ── EDA Plot 3: Top 20 users by file access volume ────────────
top_file_users = (
    files.groupby('user_id')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='file_events')
)
top_file_users = top_file_users.merge(
    ldap[['user_id'] + [c for c in ['role', 'department'] if c in ldap.columns]],
    on='user_id', how='left'
)

color_col = 'department' if 'department' in top_file_users.columns else 'user_id'

fig = px.bar(
    top_file_users,
    x='file_events', y='user_id',
    color=color_col,
    orientation='h',
    title='Top 20 Users by Total File Access Events',
    labels={'file_events': 'Total File Events', 'user_id': 'User ID'},
    height=500,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
    font_color='white', title_font_size=14,
    yaxis={'categoryorder': 'total ascending'},
)
fig.show()
print('✅ Top file access users plotted.')

In [ ]:
# ── EDA Plot 4: After-hours login rate by role ────────────────
if 'role' in logon.columns:
    logon_role = (
        logon.groupby(['role', 'is_after_hours'])
        .size()
        .reset_index(name='count')
    )
    logon_role['pct'] = logon_role.groupby('role')['count'].transform(
        lambda x: x / x.sum() * 100
    )
    after_hours_role = (
        logon_role[logon_role['is_after_hours'] == True]
        .sort_values('pct', ascending=False)
    )
    fig = px.bar(
        after_hours_role,
        x='role', y='pct',
        title='After-Hours Login Rate by Role (%)',
        labels={'pct': 'After-Hours Login %', 'role': 'Role'},
        color='pct',
        color_continuous_scale='Reds',
        height=400,
    )
    fig.update_layout(
        plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
        font_color='white', title_font_size=14,
        xaxis_tickangle=-30,
    )
    fig.show()
    print('✅ After-hours activity by role plotted.')
    print('\n💡 Roles with legitimately high after-hours rates need')
    print('   role-adjusted baselines to avoid false positives.')
else:
    print('⚠️  Role column not found in LDAP — skipping this plot.')

In [ ]:
# ── EDA Plot 5: USB event distribution ───────────────────────
usb_by_user = (
    device.groupby('user_id')
    .size()
    .reset_index(name='usb_events')
    .sort_values('usb_events', ascending=False)
)

fig = px.histogram(
    usb_by_user,
    x='usb_events',
    nbins=50,
    title='Distribution of USB Events per User',
    labels={'usb_events': 'Total USB Events'},
    color_discrete_sequence=['#FF6B35'],
    height=350,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
    font_color='white', title_font_size=14,
)
fig.show()

users_with_usb    = len(usb_by_user)
users_without_usb = len(all_users) - users_with_usb
print(f'\n💡 Users with 0 USB events  : {users_without_usb:,}')
print(f'   Users with >10 USB events: {(usb_by_user["usb_events"] > 10).sum():,}')
print('   → First-ever USB event is a HIGH-value anomaly signal.')

---
## 1.4 Feature Engineering

We transform raw event logs into a **per-user per-day behavioral feature matrix**.
Each row = one user on one day.

| Group | Features | Source |
|-------|----------|--------|
| 🕒 Temporal | After-hours ratio, session span, weekend activity | logon |
| 📁 File Access | Volume, sensitivity ratio, delete ratio, bulk flag | file |
| 💾 Device | USB count, bytes written, new device flag | device |
| 📧 Email | External ratio, attachment count, bulk send flag | email |
| 🌐 Web | Job site visits, cloud storage, unique domains | http |
| 📈 Velocity | 7-day rolling delta vs. 30-day personal baseline | all |
| 👥 Peer Cohort | Z-score vs. role-cohort mean | all + LDAP |

In [ ]:
# ── Build daily user-level aggregation base ───────────────────
print('🔧 Building daily user-level aggregation base...')

all_user_dates = pd.concat([
    logon[['user_id',  'date_only']],
    device[['user_id', 'date_only']],
    files[['user_id',  'date_only']],
    email[['user_id',  'date_only']],
    http[['user_id',   'date_only']],
]).drop_duplicates()

ldap_cols = ['user_id'] + [c for c in ['role', 'department'] if c in ldap.columns]
base = (
    all_user_dates
    .merge(ldap[ldap_cols], on='user_id', how='left')
    .sort_values(['user_id', 'date_only'])
    .reset_index(drop=True)
)

print(f'   ✅ Base frame: {len(base):,} user-day rows')
print(f'      Unique users : {base["user_id"].nunique():,}')
print(f'      Unique days  : {base["date_only"].nunique():,}')
base.head(3)

In [ ]:
# ── Feature Group 1: Temporal (from logon) ────────────────────
print('🕒 Computing temporal features...')

pc_col = 'pc' if 'pc' in logon.columns else 'user_id'

logon_daily = (
    logon
    .groupby(['user_id', 'date_only'])
    .agg(
        login_count        = ('date', 'count'),
        after_hours_logins = ('is_after_hours', 'sum'),
        weekend_logins     = ('is_weekend', 'sum'),
        first_login_hour   = ('hour', 'min'),
        last_login_hour    = ('hour', 'max'),
        unique_pcs         = (pc_col, 'nunique'),
    )
    .reset_index()
)
logon_daily['after_hours_ratio'] = (
    logon_daily['after_hours_logins'] /
    logon_daily['login_count'].clip(lower=1)
)
logon_daily['session_span_hours'] = (
    logon_daily['last_login_hour'] - logon_daily['first_login_hour']
).clip(lower=0)

print(f'   ✅ Temporal: {logon_daily.shape[1]-2} features '
      f'for {logon_daily["user_id"].nunique():,} users')

In [ ]:
# ── Feature Group 2: File Access ──────────────────────────────
print('📁 Computing file access features...')

fname_col = next(
    (c for c in files.columns if 'file' in c.lower() or 'name' in c.lower()),
    None
)
act_col = next(
    (c for c in files.columns if 'activ' in c.lower() or 'action' in c.lower()),
    None
)

def is_sensitive(path):
    if not isinstance(path, str):
        return False
    p = path.lower()
    return any(kw in p for kw in SENSITIVE_KEYWORDS)

files['is_sensitive'] = files[fname_col].apply(is_sensitive) \
    if fname_col else False

if act_col:
    files['is_delete'] = files[act_col].str.lower().str.contains(
        'delete|remove', na=False)
    files['is_copy']   = files[act_col].str.lower().str.contains(
        'copy|move', na=False)
else:
    files['is_delete'] = False
    files['is_copy']   = False

agg_file = {
    'files_accessed'  : ('user_id', 'count'),
    'sensitive_files' : ('is_sensitive', 'sum'),
    'delete_events'   : ('is_delete', 'sum'),
    'copy_events'     : ('is_copy', 'sum'),
}
if fname_col:
    agg_file['unique_directories'] = (
        fname_col,
        lambda x: x.dropna().apply(
            lambda p: os.path.dirname(str(p))
        ).nunique()
    )

files_daily = (
    files
    .groupby(['user_id', 'date_only'])
    .agg(**agg_file)
    .reset_index()
)
files_daily['sensitive_file_ratio'] = (
    files_daily['sensitive_files'] /
    files_daily['files_accessed'].clip(lower=1)
)
files_daily['delete_ratio'] = (
    files_daily['delete_events'] /
    files_daily['files_accessed'].clip(lower=1)
)
files_daily['bulk_download_flag'] = (
    files_daily['files_accessed'] > 50
).astype(int)

print(f'   ✅ File: {files_daily.shape[1]-2} features '
      f'for {files_daily["user_id"].nunique():,} users')

In [ ]:
# ── Feature Group 3: Device / USB ─────────────────────────────
print('💾 Computing device/USB features...')

bytes_col = next(
    (c for c in device.columns
     if 'byte' in c.lower() or 'size' in c.lower()),
    None
)

agg_dev = {
    'usb_event_count' : ('user_id', 'count'),
    'after_hours_usb' : ('is_after_hours', 'sum'),
}
if bytes_col:
    agg_dev['usb_bytes_written'] = (bytes_col, 'sum')

device_daily = (
    device
    .groupby(['user_id', 'date_only'])
    .agg(**agg_dev)
    .reset_index()
)

if 'usb_bytes_written' not in device_daily.columns:
    device_daily['usb_bytes_written'] = 0.0
device_daily['usb_bytes_gb'] = device_daily['usb_bytes_written'] / 1e9

# Flag first-ever USB event per user
first_usb = (
    device.groupby('user_id')['date_only']
    .min().reset_index()
    .rename(columns={'date_only': 'first_usb_date'})
)
device_daily = device_daily.merge(first_usb, on='user_id', how='left')
device_daily['new_device_flag'] = (
    device_daily['date_only'] == device_daily['first_usb_date']
).astype(int)
device_daily.drop(columns=['first_usb_date'], inplace=True)

print(f'   ✅ Device: {device_daily.shape[1]-2} features '
      f'for {device_daily["user_id"].nunique():,} users')

In [ ]:
# ── Feature Group 4: Email ────────────────────────────────────
print('📧 Computing email features...')

to_col  = next((c for c in email.columns if c.lower() == 'to'), None)
att_col = next(
    (c for c in email.columns
     if 'attach' in c.lower() or 'size' in c.lower()),
    None
)

if to_col:
    email['is_external'] = ~email[to_col].str.contains(
        r'@dtaa\.com|@company\.com', na=False, regex=True
    )
else:
    email['is_external'] = False

email['has_attachment'] = (
    email[att_col].fillna(0).astype(float) > 0
) if att_col else False

email_daily = (
    email
    .groupby(['user_id', 'date_only'])
    .agg(
        email_count             = ('user_id', 'count'),
        external_emails         = ('is_external', 'sum'),
        emails_with_attachments = ('has_attachment', 'sum'),
    )
    .reset_index()
)
email_daily['external_email_ratio'] = (
    email_daily['external_emails'] /
    email_daily['email_count'].clip(lower=1)
)
email_daily['bulk_send_flag'] = (
    email_daily['email_count'] > 30
).astype(int)

print(f'   ✅ Email: {email_daily.shape[1]-2} features '
      f'for {email_daily["user_id"].nunique():,} users')

In [ ]:
# ── Feature Group 5: Web / HTTP ───────────────────────────────
print('🌐 Computing web/HTTP features...')

url_col = next(
    (c for c in http.columns if 'url' in c.lower()),
    next((c for c in http.columns if 'domain' in c.lower()), None)
)

def classify_url(url):
    if not isinstance(url, str):
        return 0, 0, 0
    u = url.lower()
    return (
        int(any(d in u for d in JOB_SITE_DOMAINS)),
        int(any(d in u for d in CLOUD_STORAGE_DOMAINS)),
        int(any(d in u for d in WEBMAIL_DOMAINS)),
    )

if url_col:
    http[['is_job_site', 'is_cloud_storage', 'is_webmail']] = (
        http[url_col]
        .apply(classify_url)
        .apply(pd.Series)
    )
else:
    http['is_job_site'] = 0
    http['is_cloud_storage'] = 0
    http['is_webmail'] = 0

agg_http = {
    'http_requests'        : ('user_id', 'count'),
    'job_site_visits'      : ('is_job_site', 'sum'),
    'cloud_storage_visits' : ('is_cloud_storage', 'sum'),
    'webmail_visits'       : ('is_webmail', 'sum'),
}
if url_col:
    agg_http['unique_domains'] = (
        url_col,
        lambda x: x.dropna().apply(
            lambda u: u.split('/')[2] if '://' in str(u) else str(u)
        ).nunique()
    )

http_daily = (
    http
    .groupby(['user_id', 'date_only'])
    .agg(**agg_http)
    .reset_index()
)

print(f'   ✅ Web: {http_daily.shape[1]-2} features '
      f'for {http_daily["user_id"].nunique():,} users')

In [ ]:
# ── Merge all feature groups into unified daily matrix ────────
print('🔗 Merging all feature groups...')

feature_matrix = base.copy()

for daily_df, name in [
    (logon_daily,  'temporal'),
    (files_daily,  'file'),
    (device_daily, 'device'),
    (email_daily,  'email'),
    (http_daily,   'web'),
]:
    feature_matrix = feature_matrix.merge(
        daily_df, on=['user_id', 'date_only'], how='left'
    )
    print(f'   ✅ Merged {name:10s} → shape: {feature_matrix.shape}')

# Fill NaN with 0 (no activity in that source on that day)
num_cols = feature_matrix.select_dtypes(include=[np.number]).columns
feature_matrix[num_cols] = feature_matrix[num_cols].fillna(0)

print(f'\n📐 Feature matrix shape: {feature_matrix.shape}')

In [ ]:
# ── Feature Group 6: Velocity (7-day vs. 30-day baseline) ─────
print('📈 Computing velocity features...')

VELOCITY_FEATURES = [
    'files_accessed', 'email_count', 'usb_event_count',
    'after_hours_ratio', 'job_site_visits',
    'cloud_storage_visits', 'external_email_ratio',
    'sensitive_file_ratio',
]
vel_present = [f for f in VELOCITY_FEATURES if f in feature_matrix.columns]

feature_matrix = feature_matrix.sort_values(['user_id', 'date_only'])

for feat in tqdm(vel_present, desc='Velocity'):
    grp = feature_matrix.groupby('user_id')[feat]
    feature_matrix[f'{feat}_baseline30'] = grp.transform(
        lambda x: x.rolling(BASELINE_WINDOW_DAYS, min_periods=5)
                   .mean().shift(1)
    )
    feature_matrix[f'{feat}_roll7'] = grp.transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
    feature_matrix[f'{feat}_velocity'] = (
        (feature_matrix[f'{feat}_roll7'] -
         feature_matrix[f'{feat}_baseline30']) /
        feature_matrix[f'{feat}_baseline30'].abs().clip(lower=0.01)
    ).clip(-10, 10)

print(f'   ✅ Velocity features added: {len(vel_present) * 3} new columns')

In [ ]:
# ── Feature Group 7: Peer Cohort Z-Score ─────────────────────
print('👥 Computing peer cohort features...')

COHORT_FEATURES = [
    'files_accessed', 'email_count', 'usb_event_count',
    'after_hours_ratio', 'external_email_ratio',
]
cohort_present = [
    f for f in COHORT_FEATURES
    if f in feature_matrix.columns and 'role' in feature_matrix.columns
]

for feat in tqdm(cohort_present, desc='Cohort z-scores'):
    cohort_stats = (
        feature_matrix
        .groupby(['role', 'date_only'])[feat]
        .agg(cohort_mean='mean', cohort_std='std')
        .reset_index()
    )
    cohort_stats['cohort_std'] = (
        cohort_stats['cohort_std'].fillna(1).clip(lower=0.01)
    )
    feature_matrix = feature_matrix.merge(
        cohort_stats, on=['role', 'date_only'], how='left'
    )
    feature_matrix[f'{feat}_cohort_zscore'] = (
        (feature_matrix[feat] - feature_matrix['cohort_mean']) /
        feature_matrix['cohort_std']
    ).clip(-5, 5)
    feature_matrix.drop(
        columns=['cohort_mean', 'cohort_std'], inplace=True
    )

if not cohort_present:
    print('   ⚠️  Skipped — role column not available in LDAP.')
else:
    print(f'   ✅ Cohort z-scores added: {len(cohort_present)} features')

print(f'\n📐 Final feature matrix shape: {feature_matrix.shape}')

In [ ]:
# ── Feature matrix summary ────────────────────────────────────
num_cols = feature_matrix.select_dtypes(include=[np.number]).columns.tolist()

print('=' * 60)
print('  FEATURE MATRIX SUMMARY')
print('=' * 60)
print(f'  Total rows (user-days)  : {len(feature_matrix):,}')
print(f'  Unique users            : {feature_matrix["user_id"].nunique():,}')
print(f'  Unique days             : {feature_matrix["date_only"].nunique():,}')
print(f'  Total numeric features  : {len(num_cols)}')
print(f'  Missing values          : '
      f'{feature_matrix[num_cols].isna().sum().sum():,}')
print('=' * 60)

groups = {
    'Temporal'   : [c for c in num_cols if any(k in c for k in
                    ['login', 'after_hours', 'session', 'weekend', 'hour'])],
    'File'       : [c for c in num_cols if any(k in c for k in
                    ['file', 'sensitive', 'delete', 'copy', 'bulk', 'dir'])],
    'Device/USB' : [c for c in num_cols if 'usb' in c or 'device' in c],
    'Email'      : [c for c in num_cols if 'email' in c or 'attach' in c],
    'Web'        : [c for c in num_cols if any(k in c for k in
                    ['http', 'job_site', 'cloud', 'webmail', 'domain'])],
    'Velocity'   : [c for c in num_cols if 'velocity' in c],
    'Cohort'     : [c for c in num_cols if 'cohort' in c],
}
print('\n  Feature breakdown by group:')
for grp, cols in groups.items():
    print(f'  {grp:<12}: {len(cols):>3} features')

In [ ]:
# ── Behavioral fingerprint heatmap (top 30 users) ─────────────
sample_users = (
    feature_matrix.groupby('user_id')['files_accessed']
    .sum()
    .sort_values(ascending=False)
    .head(30)
    .index.tolist()
)

viz_features = [
    'files_accessed', 'after_hours_ratio', 'usb_event_count',
    'external_email_ratio', 'job_site_visits',
    'sensitive_file_ratio', 'email_count', 'cloud_storage_visits',
]
viz_present = [f for f in viz_features if f in feature_matrix.columns]

user_avg = (
    feature_matrix[feature_matrix['user_id'].isin(sample_users)]
    .groupby('user_id')[viz_present]
    .mean()
)
user_avg_norm = (
    (user_avg - user_avg.min()) /
    (user_avg.max() - user_avg.min() + 1e-9)
)

fig = px.imshow(
    user_avg_norm,
    title='Behavioral Fingerprint Heatmap — Top 30 Users (Normalised)',
    labels={'x': 'Feature', 'y': 'User ID', 'color': 'Normalised Value'},
    color_continuous_scale='RdYlGn_r',
    aspect='auto',
    height=600,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
    font_color='white', title_font_size=13,
)
fig.show()
print('✅ Behavioral fingerprint heatmap rendered.')
print('💡 Red = high activity. Users with many red cells across')
print('   multiple features are candidates for anomaly investigation.')

---
## 1.5 Save Feature Matrix

In [ ]:
# ── Save feature matrix ───────────────────────────────────────
parquet_path = 'feature_matrix.parquet'
csv_path     = 'feature_matrix.csv'

feature_matrix.to_parquet(parquet_path, index=False)
feature_matrix.to_csv(csv_path, index=False)

print('=' * 60)
print('  ✅ Feature matrix saved!')
print(f'     Parquet : {parquet_path}')
print(f'     CSV     : {csv_path}')
print(f'     Shape   : {feature_matrix.shape}')
print(f'     Size    : {os.path.getsize(parquet_path) / 1e6:.1f} MB (parquet)')
print('=' * 60)
print('\n🎯 Notebook 01 Complete!')
print('   → Open 02_ml_anomaly_detection.ipynb to train the ML models.')

---
## ✅ Notebook 01 — Summary

### What We Built
| Step | Output |
|------|--------|
| Extracted `r4.2.tar.bz2` | `./cert_r42/` directory |
| Loaded 5 CERT event log CSVs | Raw event dataframes |
| Enriched with LDAP org data | Role + department per user |
| Explored behavioral patterns | 5 EDA visualisations |
| Engineered 7 feature groups | Temporal, File, Device, Email, Web, Velocity, Cohort |
| Saved feature matrix | `feature_matrix.parquet` + `feature_matrix.csv` |

### Key Observations
- 🕒 After-hours activity is rare for most roles → strong anomaly signal
- 💾 USB events are near-zero for most users → first event is highly suspicious
- 📁 Sensitive file access is concentrated in a small subset of users
- 👥 Role-adjusted baselines are essential to avoid false positives

### Next Step
**Notebook 02** — Train Isolation Forest, Autoencoder, and LSTM on this
feature matrix to generate per-user anomaly scores.